# Network Security Project: Visualizations & Evaluation Metrics

This notebook focuses exclusively on generating high-quality graphs, charts, and mathematical evaluation matrices for your DoH C2 Detection project. You can copy/paste these directly into your final academic report or presentation.

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn pyarrow
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc, RocCurveDisplay
import os

sns.set_theme(style="whitegrid", palette="pastel")
%matplotlib inline

## 1. Load Data & Train the Evaluator Model
We load our generated simulated dataset (`labeled_doh_dataset.csv`) to plot our metrics against.

In [ ]:
# Ensure the dataset exists from the main simulation notebook
if not os.path.exists('labeled_doh_dataset.csv'):
    raise FileNotFoundError("Please make sure 'labeled_doh_dataset.csv' is uploaded to Colab!")

df = pd.read_csv('labeled_doh_dataset.csv')
drop_cols = ['src_ip', 'dst_ip', 'src_mac', 'dst_mac', 'src_port', 'dst_port', 'timestamp', 'flow_id']
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True, errors='ignore')
df.replace([np.inf, -np.inf, 'Infinity'], np.nan, inplace=True)
df.dropna(inplace=True)

X = df.drop(columns=['Label'])
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1] # Probabilities for ROC curve

print("Model trained and ready for Visualization Construction!")

## 2. Dataset Distribution Mapping
A pie chart showing the exact class imbalance between Malicious C2 and Benign Traffic.

In [ ]:
plt.figure(figsize=(7, 7))
counts = y.value_counts()
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140, colors=['#ff9999','#66b3ff'])
plt.title('Dataset Ground Truth Distribution (Benign vs Malicious DoH)', fontsize=14, fontweight='bold')
plt.show()

## 3. The Confusion Matrix (Evaluation Matrix)
The standard matrix for Network Security. It shows True Positives, True Negatives, False Positives, and False Negatives.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = model.classes_

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, annot_kws={"size": 16})
plt.xlabel('Predicted by ML', fontsize=12)
plt.ylabel('Actual Truth', fontsize=12)
plt.title('Evaluation Confusion Matrix: C2 Detection', fontsize=14, fontweight='bold')
plt.show()

## 4. Evaluation Matrices: Precision, Recall, F1-Score
Calculates the exact harmonic mathematical evaluation metrics of your classifier.

In [ ]:
print("---------------------------------------------------------")
print("                 PROJECT EVALUATION MATRIX               ")
print("---------------------------------------------------------")
print(f"Overall Accuracy Score: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\nDetailed Classification Report:")
report = classification_report(y_test, y_pred)
print(report)

## 5. ROC Curve & AUC Score
The Receiver Operating Characteristic (ROC) curve plots the True Positive Rate against the False Positive Rate. A curve hugging the top left corner (AUC near 1.0) indicates a highly perfect C2 detecting model.

In [ ]:
# Convert string labels to binary for ROC calculation (e.g. Malicious = 1, Benign = 0)
y_test_binary = np.where(y_test == 'Malicious', 1, 0)
fpr, tpr, thresholds = roc_curve(y_test_binary, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) - DoH Analysis', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.show()

## 6. Feature Importances Graph
A mandatory visualization for any ML-based security project. It explicitly proves to reviewers *how* the AI caught the Malware. We map this back to Flow Duration, Payload Lengths, and IAT.

In [ ]:
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importances.head(15), palette='magma')
plt.title('Top 15 Detection Characteristics (Feature Importance)', fontsize=14, fontweight='bold')
plt.xlabel('Significance to AI', fontsize=12)
plt.ylabel('Network Feature', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Flow Behavioral Variances (Density Scatters)
This plots the physical difference in traffic shapes between a Human Browser (Benign) and Malware (Malicious) based on Inter-Arrival Times and Packet Sizes. Evasion attacks attempt to overlap these dots.

In [ ]:
try:
    # We try to grab standard CICFlowMeter payload mappings (names differ sometimes depending on version)
    time_metric = 'flow_duration' if 'flow_duration' in df.columns else 'Duration'
    size_metric = 'totlen_fwd_pkts' if 'totlen_fwd_pkts' in df.columns else 'FlowBytesSent'

    if time_metric in df.columns and size_metric in df.columns:
        plt.figure(figsize=(9, 6))
        sns.scatterplot(data=df, x=time_metric, y=size_metric, hue="Label", palette={'Benign': 'green', 'Malicious': 'red'}, alpha=0.7, s=60)
        
        plt.title('Behavioral Scatter: Flow Time vs Forward Payload Size', fontsize=14, fontweight='bold')
        plt.xlabel('Connection Duration (Sec)', fontsize=12)
        plt.ylabel('Total Forward Bytes', fontsize=12)
        plt.show()
    else:
        print("Required dimension columns missing for Scatter Plotting.")
except Exception as e:
    print("Scatter error:", e)